In [1]:
import yfinance as yf
import pandas as pd
import ssl
import time
from datetime import datetime, timedelta
import requests
import io

# descargo datos desde 2018 para coincidirlo con el indice F&G 
btc = yf.download("BTC-USD", start="2018-02-01", auto_adjust=False)
ssl._create_default_https_context = ssl._create_unverified_context

# reseteo el indice, dejo de tener date como indice y pasa a ser columna
btc.reset_index(inplace=True)

# Filtro solo las columnas necesarias
btc = btc[["Date", "Close", "High", "Low", "Open", "Volume"]]

# guardo csv base
btc.to_csv("../Data/btc.csv", index=False)

btc.tail() ## Inspecciono las ultimas filas del DF para corroborrar que las columnas esten bien


[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
Ticker,,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
3108,2026-08-06,64262.113281,64934.492188,64098.476562,64595.449219,18529402711
3109,2026-08-07,64880.191406,65330.609375,64113.273438,64257.488281,22165720102
3110,2026-08-08,64904.687500,65140.480469,64797.113281,64882.546875,12350094271
3111,2026-08-09,64844.886719,65401.691406,64677.601562,64906.550781,13234538380
3112,2026-08-10,64043.179688,65278.343750,63764.757812,64848.906250,23706193920


In [2]:
#Indice Fear and Greed -- Mide el miedl del mercado 

url = "https://api.alternative.me/fng/?date_format=%2701%2F01%2F2018%27&format=csv&limit=50000"
response = requests.get(url)

if response.status_code == 200:
    content = response.content.decode('utf-8')
    start = content.find("fng_value")
    csv_data = content[start:]
    # Leer el CSV desde el string
    df = pd.read_csv(io.StringIO(csv_data))
    # Renombrar columnas
    df = df.rename(columns={
        'fng_value': 'date',
        'fng_classification': 'fng_value',
        'date': 'fng_classification'
    })
    # Guardar el nuevo CSV
    df = df.iloc[:-5]
    df.to_csv("../Data/fear_greed.csv", index=False)
    print(df.head(20)) #Muestro los ultimos 20 datos 
else:
    print("Error:", response.status_code)

          date  fng_value fng_classification
0   10-08-2026       30.0               Fear
1   09-08-2026       31.0               Fear
2   08-08-2026       30.0               Fear
3   07-08-2026       29.0               Fear
4   06-08-2026       25.0       Extreme Fear
5   05-08-2026       27.0               Fear
6   04-08-2026       25.0       Extreme Fear
7   03-08-2026       28.0               Fear
8   02-08-2026       27.0               Fear
9   01-08-2026       27.0               Fear
10  31-07-2026       25.0       Extreme Fear
11  30-07-2026       28.0               Fear
12  29-07-2026       29.0               Fear
13  28-07-2026       29.0               Fear
14  27-07-2026       30.0               Fear
15  26-07-2026       26.0               Fear
16  25-07-2026       27.0               Fear
17  24-07-2026       28.0               Fear
18  23-07-2026       31.0               Fear
19  22-07-2026       33.0               Fear
